In [5]:
#!/usr/bin/env python3
"""
Compare PlayerBoxScores.parquet and PlayerStatus.parquet across S3.
Find all instances, compare ETags, and report if they're identical.
"""
!aws login

import boto3
import logging
from collections import defaultdict
from datetime import datetime

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('s3_duplicate_check.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

BUCKET = 'nba-265753586044-us-east-1-an'
TARGET_FILES = ['PlayerBoxScores.parquet', 'PlayerStatus.parquet']

def scan_s3_for_files():
    """Recursively scan S3 for target files, return {filename: [locations]}"""
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')

    files_found = defaultdict(list)

    try:
        logger.info(f"Scanning s3://{BUCKET}/ recursively...")
        for page in paginator.paginate(Bucket=BUCKET):
            if 'Contents' not in page:
                continue

            for obj in page['Contents']:
                key = obj['Key']
                for target in TARGET_FILES:
                    if key.endswith(target):
                        files_found[target].append({
                            'key': key,
                            'etag': obj['ETag'].strip('"'),
                            'size': obj['Size'],
                            'last_modified': obj['LastModified'].isoformat()
                        })

        logger.info(f"Found {sum(len(v) for v in files_found.values())} files")
        return files_found

    except Exception as e:
        logger.error(f"Error scanning S3: {e}")
        raise

def compare_files(files_found):
    """Compare ETags between PlayerBoxScores and PlayerStatus"""
    logger.info("\n" + "="*80)
    logger.info("COMPARISON RESULTS")
    logger.info("="*80)

    box_scores = files_found.get('PlayerBoxScores.parquet', [])
    player_status = files_found.get('PlayerStatus.parquet', [])

    logger.info(f"\nFound {len(box_scores)} PlayerBoxScores.parquet instances")
    logger.info(f"Found {len(player_status)} PlayerStatus.parquet instances")

    # Group by ETag
    etag_to_files = defaultdict(list)

    for f in box_scores:
        etag_to_files[f['etag']].append(('PlayerBoxScores.parquet', f))

    for f in player_status:
        etag_to_files[f['etag']].append(('PlayerStatus.parquet', f))

    logger.info(f"\n{len(etag_to_files)} unique ETags found")

    # Find matching/differing versions
    matching_pairs = []
    differing_versions = []

    for etag, files_with_etag in etag_to_files.items():
        file_types = set(f[0] for f in files_with_etag)

        if len(file_types) == 2 and len(files_with_etag) == 2:
            # Both files have same ETag
            matching_pairs.append((etag, files_with_etag))
        else:
            differing_versions.append((etag, files_with_etag))

    # Report matching pairs
    logger.info(f"\n{'MATCHING PAIRS (same content):':^80}")
    logger.info("-" * 80)
    if matching_pairs:
        for etag, files_list in matching_pairs:
            logger.info(f"ETag: {etag}")
            for fname, f in files_list:
                logger.info(f"  {fname}")
                logger.info(f"    Path: {f['key']}")
                logger.info(f"    Size: {f['size']} bytes | Modified: {f['last_modified']}")
            logger.info("")
    else:
        logger.info("No matching pairs found - files are always different!")

    # Report differences
    logger.info(f"\n{'DIFFERENT VERSIONS':^80}")
    logger.info("-" * 80)
    if differing_versions:
        for etag, files_list in differing_versions:
            logger.info(f"ETag: {etag}")
            for fname, f in files_list:
                logger.info(f"  {fname}")
                logger.info(f"    Path: {f['key']}")
                logger.info(f"    Size: {f['size']} bytes | Modified: {f['last_modified']}")
            logger.info("")

    # Summary
    logger.info("\n" + "="*80)
    logger.info("SUMMARY")
    logger.info("="*80)
    logger.info(f"Matching pairs (identical files): {len(matching_pairs)}")
    logger.info(f"Differing versions: {len(differing_versions)} unique hashes")

    if len(matching_pairs) == 0 and len(box_scores) > 0 and len(player_status) > 0:
        logger.warning("\n⚠️   WARNING: PlayerBoxScores and PlayerStatus are NEVER identical!")
        logger.warning("These appear to be different datasets.")
    elif len(matching_pairs) > 0 and len(differing_versions) == 0:
        logger.info("\n✓ All versions are identical - these files are duplicates!")
    else:
        logger.info("\n⚠️   Files are sometimes identical, sometimes different")

if __name__ == '__main__':
    logger.info(f"Starting S3 duplicate check at {datetime.now()}")
    files_found = scan_s3_for_files()
    compare_files(files_found)
    logger.info(f"\nCheck complete. See s3_duplicate_check.log for full details.")


Attempting to open your default browser.
If the browser does not open, open the following URL:

https://us-east-1.signin.aws.amazon.com/v1/authorize?response_type=code&client_id=arn%3Aaws%3Asignin%3A%3A%3Adevtools%2Fsame-device&state=13c1244a-a492-4850-bb26-81ad4ef9cba1&code_challenge_method=SHA-256&scope=openid&redirect_uri=http%3A%2F%2F127.0.0.1%3A50768%2Foauth%2Fcallback&code_challenge=gw9X_cTbMZ23HQkqMFRDn74zlnowAPsLfF7qh7ULu8o

Updated profile default to use arn:aws:iam::265753586044:user/admin credentials.


2026-06-02 15:51:04,326 - INFO - Starting S3 duplicate check at 2026-06-02 15:51:04.326592
2026-06-02 15:51:04,361 - INFO - Scanning s3://nba-265753586044-us-east-1-an/ recursively...
2026-06-02 15:52:18,283 - INFO - Found 18 files
2026-06-02 15:52:18,286 - INFO - 
2026-06-02 15:52:18,286 - INFO - COMPARISON RESULTS
2026-06-02 15:52:18,286 - INFO - ================================================================================
2026-06-02 15:52:18,287 - INFO - 
Found 9 PlayerBoxScores.parquet instances
2026-06-02 15:52:18,287 - INFO - Found 9 PlayerStatus.parquet instances
2026-06-02 15:52:18,288 - INFO - 
1 unique ETags found
2026-06-02 15:52:18,288 - INFO - 
                         MATCHING PAIRS (same content):                         
2026-06-02 15:52:18,288 - INFO - --------------------------------------------------------------------------------
2026-06-02 15:52:18,289 - INFO - No matching pairs found - files are always different!
2026-06-02 15:52:18,289 - INFO - 
               

In [9]:
# check_live_games_direct.py
import logging
import requests
import json

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

def check_live_and_recent_games_direct():
    logger.info("Bypassing library wrapper; querying cdn.nba.com directly...")
    
    # Direct CDN scoreboard endpoint used for active real-time data
    url = "https://cdn.nba.com/static/json/liveData/scoreboard/todaysScoreboard_00.json"
    
    # Mirror complete modern browser headers to clear CDN protection walls
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "en-US,en;q=0.9",
        "Origin": "https://www.nba.com",
        "Referer": "https://www.nba.com/",
        "Connection": "keep-alive"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        
        # Guard against non-200 responses safely
        if response.status_code != 200:
            logger.error(f"CDN Endpoint returned status code: {response.status_code}. Data may not be ready.")
            return

        # Safely parse the raw payload text
        try:
            score_dict = response.json()
        except ValueError as e:
            logger.error("Failed to parse JSON. Raw body snippet: " + response.text[:200])
            raise e
            
        scoreboard_data = score_dict.get("scoreboard", {})
        game_date = scoreboard_data.get("gameDate")
        games = scoreboard_data.get("games", [])
        
        print(f"\nDirect Live CDN Scoreboard Date Context: {game_date}")
        print(f"{'='*65}")
        
        if not games:
            print("No active or scheduled games found on the live slate for this frame.")
            return
            
        for game in games:
            game_id = game.get("gameId")
            game_status_text = game.get("gameStatusText", "").strip()
            
            home_team = game.get("homeTeam", {})
            away_team = game.get("awayTeam", {})
            
            home_tricode = home_team.get("teamTricode")
            away_tricode = away_team.get("teamTricode")
            
            home_score = home_team.get("score", 0)
            away_score = away_team.get("score", 0)
            
            print(f"Game ID: {game_id} | Status: {game_status_text:12s} | Matchup: {away_tricode} {away_score} @ {home_tricode} {home_score}")
            
    except Exception as e:
        logger.error(f"Failed to extract real-time scoreboard matrix: {e}", exc_info=True)

if __name__ == "__main__":
    check_live_and_recent_games_direct()

2026-06-08 21:06:16,232 - INFO - Bypassing library wrapper; querying cdn.nba.com directly...



Direct Live CDN Scoreboard Date Context: 2026-06-08
Game ID: 0042500403 | Status: Final        | Matchup: SAS 115 @ NYK 111


In [24]:
import sys
from nba_api.stats.endpoints import hustlestatsboxscore

def run_sanity_check():
    # A verified completed game from the 2026 Finals
    completed_game_id = "0042500402"
    
    # The target unplayed game ID
    future_game_id = "0042500403"
    
    print("--- STEP 1: Querying Historical Game Data ---")
    try:
        print(f"Requesting Hustle Stats for Game: {completed_game_id}...")
        # Using the correct endpoint class
        historical_call = hustlestatsboxscore.HustleStatsBoxScore(game_id=completed_game_id)
        data = historical_call.get_dict()
        
        # Extract the core data sets
        result_sets = data.get('resultSets', [])
        
        # Check the primary dataset (PlayerStats / TeamStats)
        if result_sets and len(result_sets[1].get('rowSet', [])) > 0:
            print(f"Success! Historical data returned active tracking records.")
            print(f"Sample data columns: {result_sets[1].get('headers')[:5]}")
        else:
            print("Warning: API responded but data tables are empty.")
            
    except Exception as e:
        print(f"CRITICAL ERROR: Historical game unexpectedly failed: {e}")
        sys.exit(1)

    print("\n--- STEP 2: Querying Future/Missing Game Data (Sanity Check) ---")
    try:
        print(f"Requesting Hustle Stats for Game: {future_game_id}...")
        future_call = hustlestatsboxscore.HustleStatsBoxScore(game_id=future_game_id)
        raw_json = future_call.get_dict()
        
        # Detect if the backend returns a shell without any game data entries
        if not raw_json.get('resultSets') or len(raw_json['resultSets'][1].get('rowSet', [])) == 0:
            raise ValueError("NBA API returned an empty payload wrapper for unplayed game.")
            
        print("SANITY CHECK FAILED: The API successfully returned active data for a future game ID.")
        
    except Exception as e:
        print("SANITY CHECK PASSED: The application errored out as expected!")
        print(f"Caught Expected Exception Type/Message: {e}")

if __name__ == "__main__":
    run_sanity_check()

--- STEP 1: Querying Historical Game Data ---
Requesting Hustle Stats for Game: 0042500402...
Success! Historical data returned active tracking records.
Sample data columns: ['GAME_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_CITY', 'PLAYER_ID']

--- STEP 2: Querying Future/Missing Game Data (Sanity Check) ---
Requesting Hustle Stats for Game: 0042500403...
SANITY CHECK FAILED: The API successfully returned active data for a future game ID.


In [23]:
from nba_api.stats.endpoints import BoxScoreSummaryV3
# Get game summary
summary = BoxScoreSummaryV3(game_id="0022500142")

# Access game info as dictionary
game_info = summary.game_info.get_dict()
# Access team statistics as DataFrame
stats_df = summary.other_stats.get_data_frame()
print(stats_df.columns)

Index(['gameId', 'teamId', 'teamCity', 'teamName', 'teamTricode', 'points',
       'reboundsTotal', 'assists', 'steals', 'blocks', 'turnovers',
       'fieldGoalsPercentage', 'threePointersPercentage',
       'freeThrowsPercentage', 'pointsInThePaint', 'pointsSecondChance',
       'pointsFastBreak', 'biggestLead', 'leadChanges', 'timesTied',
       'biggestScoringRun', 'turnoversTeam', 'turnoversTotal', 'reboundsTeam',
       'pointsFromTurnovers', 'benchPoints'],
      dtype='str')


In [24]:
# Run this cell inside your Jupyter Notebook to diagnose the endpoint
import json
from nba_api.stats.endpoints import BoxScoreHustleV2

# Match the elaborate header fingerprint from the script
headers = {
    "Host": "stats.nba.com",
    "Connection": "keep-alive",
    "Pragma": "no-cache",
    "Cache-Control": "no-cache",
    "sec-ch-ua": '"Google Chrome";v="125", "Chromium";v="125", "Not.A/Brand";v="24"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"macOS"',
    "Upgrade-Insecure-Requests": "1",
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://www.nba.com",
    "Sec-Fetch-Site": "same-site",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Dest": "empty",
    "Referer": "https://www.nba.com/",
    "Accept-Encoding": "gzip, deflate, br",
    "Accept-Language": "en-US,en;q=0.9",
    "DNT": "1"
}

try:
    print("Testing BoxScoreHustleV2 in Jupyter...")
    hustle_test = BoxScoreHustleV2(game_id="0011900131", headers=headers, timeout=15)
    raw_json = hustle_test.nba_response.get_json()
    print("SUCCESS: Endpoint is accessible in Jupyter.")
    print(f"Payload keys: {list(json.loads(raw_json).keys())}")
except Exception as e:
    print(f"FAILED in Jupyter: {str(e)}")

Testing BoxScoreHustleV2 in Jupyter...
FAILED in Jupyter: 'NoneType' object has no attribute 'get'


In [1]:
"""Generate SCHEMA.md"""
import os
from datetime import datetime
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

def generate_schema_registry(target_dir: str, output_file: str):
    """
    Scans the target directory for CSV and Parquet files, extracts their 
    schemas efficiently, and writes a structured Markdown registry.
    """
    base_path = Path(target_dir).resolve()
    out_path = Path(output_file).resolve()
    
    if not base_path.exists():
        raise FileNotFoundError(f"Target directory {base_path} does not exist.")
        
    markdown_lines = [
        "# Data Schema Registry",
        f"**Generated on:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"**Base Directory:** `{base_path}`",
        "---",
        ""
    ]
    
    # Locate all target data files recursively
    extensions = ['*.csv', '*.parquet', '*.pq']
    files = []
    for ext in extensions:
        files.extend(base_path.rglob(ext))
        
    # Sort files for a deterministic, clean output order
    files = sorted(list(set(files)))
    
    if not files:
        markdown_lines.append("No `.csv` or `.parquet` files found within the specified directory.")
        markdown_lines.append("")
    
    for file_path in files:
        # Calculate relative path from the base directory for clean documentation
        rel_path = file_path.relative_to(base_path)
        
        markdown_lines.append(f"## File: `{rel_path}`")
        markdown_lines.append(f"- **Format:** {file_path.suffix.upper()[1:]}")
        markdown_lines.append(f"- **Location:** `{file_path}`")
        markdown_lines.append("")
        markdown_lines.append("| Column Name | Data Type |")
        markdown_lines.append("| :--- | :--- |")
        
        try:
            if file_path.suffix.lower() == '.csv':
                # Use nrows=1 to infer types instantly without memory overhead
                df = pd.read_csv(file_path, nrows=1)
                if df.empty and len(df.columns) == 0:
                    markdown_lines.append("| *Empty File* | *No columns detected* |")
                else:
                    for col, dtype in df.dtypes.items():
                        markdown_lines.append(f"| {col} | {dtype} |")
                        
            elif file_path.suffix.lower() in ['.parquet', '.pq']:
                # Read metadata schema directly from the Parquet footer
                schema = pq.read_schema(str(file_path))
                if len(schema) == 0:
                    markdown_lines.append("| *Empty Schema* | *No columns detected* |")
                else:
                    for field in schema:
                        markdown_lines.append(f"| {field.name} | {field.type} |")
                        
        except Exception as e:
            markdown_lines.append(f"| **Error Parsing Schema** | *{str(e)}* |")
            
        markdown_lines.append("")
        markdown_lines.append("---")
        markdown_lines.append("")

    # Ensure output directory structures exist before writing
    out_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(markdown_lines))
        
    print(f"Successfully generated schema registry at: {out_path}")

if __name__ == "__main__":
    TARGET_DIR = "/Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data"
    OUTPUT_FILE = os.path.join(TARGET_DIR, "SCHEMA.md")
    
    generate_schema_registry(TARGET_DIR, OUTPUT_FILE)

Successfully generated schema registry at: /Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data/SCHEMA.md


In [3]:
import os
from pathlib import Path

def rename_parquet_files(target_dir: str, dry_run: bool = True):
    """
    Scans the target directory recursively for Parquet files.
    Removes 'CRUDE' and '_' from the filename stems.
    
    Parameters:
    -----------
    target_dir : str
        The path to the root data directory.
    dry_run : bool, default True
        If True, only displays planned modifications without executing them.
        If False, permanently renames files on disk.
    """
    base_path = Path(target_dir).resolve()
    
    if not base_path.exists():
        raise FileNotFoundError(f"Target directory {base_path} does not exist.")
        
    extensions = ['*.parquet', '*.pq']
    files = []
    for ext in extensions:
        files.extend(base_path.rglob(ext))
        
    # De-duplicate file list if any extension search overlapped
    files = sorted(list(set(files)))
    
    modified_count = 0
    
    if dry_run:
        print("=" * 60)
        print("⚠️ RUNNING IN DRY RUN MODE - NO FILES WILL BE CHANGED ⚠️")
        print("=" * 60 + "\n")
    else:
        print("=" * 60)
        print("🚀 LIVE RUN - MODIFYING FILES ON DISK")
        print("=" * 60 + "\n")
    
    for file_path in files:
        stem = file_path.stem
        suffix = file_path.suffix
        
        # Check if either 'CRUDE' or '_' exists in the filename stem
        if "CRUDE" in stem or "_" in stem:
            # Strip 'CRUDE' out
            new_stem = stem.replace("CRUDE", "")
            
            # Strip underscores out
            new_stem = new_stem.replace("_", "")
            
            # Fallback if the clean name becomes completely empty
            if not new_stem:
                new_stem = "cleaned_file"
                
            new_filename = f"{new_stem}{suffix}"
            new_file_path = file_path.with_name(new_filename)
            
            # Handle edge case where target filename already exists to avoid overwriting
            counter = 1
            while new_file_path.exists() and new_file_path != file_path:
                new_file_path = file_path.with_name(f"{new_stem}_{counter}{suffix}")
                counter += 1
                
            if new_file_path != file_path:
                status_label = "[DRY RUN MATCH]" if dry_run else "[RENAMING]"
                print(f"{status_label}")
                print(f"  Old: {file_path.name}")
                print(f"  New: {new_file_path.name}")
                print(f"  Path: {file_path.parent}\n")
                
                if not dry_run:
                    file_path.rename(new_file_path)
                    
                modified_count += 1

    summary_status = "would be modified" if dry_run else "were modified"
    print(f"Process complete. Total files that {summary_status}: {modified_count}")

if __name__ == "__main__":
    TARGET_DIR = "/Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data"
    
    # Toggle dry_run=False when you are ready to apply changes permanently
    rename_parquet_files(TARGET_DIR, dry_run=False)

🚀 LIVE RUN - MODIFYING FILES ON DISK

[RENAMING]
  Old: CRUDEPlayByPlay.parquet
  New: PlayByPlay.parquet
  Path: /Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data

[RENAMING]
  Old: CRUDESummaryBroadcasters.parquet
  New: SummaryBroadcasters.parquet
  Path: /Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data

[RENAMING]
  Old: CRUDESummaryLastFive.parquet
  New: SummaryLastFive.parquet
  Path: /Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data

[RENAMING]
  Old: CRUDESummaryOfficials.parquet
  New: SummaryOfficials.parquet
  Path: /Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data

[RENAMING]
  Old: CRUDESummaryPlayers.parquet
  New: SummaryPlayers.parquet
  Path: /Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data

[RENAMING]
  Old: CRUDESummary_GameMeta.parquet
  New: SummaryGameMeta.parquet
  Path: /Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data

[RENAMING

In [1]:
import glob
import os
import pandas as pd

# Define the target directory and pattern
directory_path = "/Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data"
search_pattern = os.path.join(directory_path, "CRUDE*.parquet")

# Find all matching files
parquet_files = glob.glob(search_pattern)

if not parquet_files:
    print(f"No files matching 'CRUDE*.parquet' found in {directory_path}")
else:
    print(f"Found {len(parquet_files)} file(s). Analyzing data quality...\n")
    
    # Iterate and inspect each file
    for file_path in parquet_files:
        file_name = os.path.basename(file_path)
        print(f"--- File: {file_name} ---")
        
        try:
            # Attempt to read the parquet file to check structural health
            df = pd.read_parquet(file_path, engine="pyarrow")
            
            # Count total missing/NaN values across the entire file
            total_nans = df.isna().sum().sum()
            cols = [col for col in df.columns if len(df.columns)>20]
            print(f"  Status: Healthy")
            print(f"  Total Rows: {len(df)}")
            print(f"  Total NaN/Missing Values: {total_nans}")
            
            # If there are NaNs, break them down per column
            if total_nans > 0:
                print("  NaN breakdown by column:")
                nan_counts = df.isna().sum()
                for col, count in nan_counts[nan_counts > 0].items():
                    print(f"    - {col}: {count} missing")

            for c in cols:
                print(f"      {c}")
                    
        except Exception as e:
            # Catch file corruption, incomplete writes, or schema structural bugs
            print(f"  Status: ❌ CORRUPTED OR UNREADABLE")
            print(f"  Error Detail: {str(e)}")
        
        print("\n" + "="*50 + "\n")

Found 7 file(s). Analyzing data quality...

--- File: CRUDESummaryPlayers.parquet ---
  Status: Healthy
  Total Rows: 1172040
  Total NaN/Missing Values: 320940
  NaN breakdown by column:
    - name: 160470 missing
    - nameI: 160470 missing


--- File: CRUDEPlayByPlay.parquet ---
  Status: Healthy
  Total Rows: 18960250
  Total NaN/Missing Values: 0
      gameId
      videoAvailable_game
      actionNumber
      clock
      period
      teamId
      teamTricode
      personId
      playerName
      playerNameI
      xLegacy
      yLegacy
      shotDistance
      shotResult
      isFieldGoal
      scoreHome
      scoreAway
      pointsTotal
      location
      description
      actionType
      subType
      videoAvailable
      shotValue
      actionId


--- File: CRUDEHustle.parquet ---
  Status: Healthy
  Total Rows: 18247
  Total NaN/Missing Values: 10615775
  NaN breakdown by column:
    - boxScoreHustle_homeTeam_players_1_personId: 4798 missing
    - boxScoreHustle_homeTeam_pla

In [ ]:
import re
import pandas as pd

PATH    = '/Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data/CRUDEHustle.parquet'
OUT_DIR = '/Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data/'

df = pd.read_parquet(PATH)
df = df.drop(columns=[c for c in df.columns if c.startswith('meta_')])

# ── Rename columns ────────────────────────────────────────────────────────────
# Target format for player cols: {stat}.{side}.{slot}
# e.g. boxScoreHustle_homeTeam_players_3_statistics_deflections
#   →  deflections.homeTeam.3
# wide_to_long matches: stubname + sep + j-value
# so stub=deflections, sep='.', j=(homeTeam.3)

PLAYER_RE = re.compile(
    r'^boxScoreHustle_(homeTeam|awayTeam)_players_(\d+)_(?:statistics_)?(.+)$'
)

def rename(col):
    m = PLAYER_RE.match(col)
    if m:
        side, slot, stat = m.group(1), m.group(2), m.group(3)
        return f"{stat}.{side}.{slot}"   # stat is the stub
    return col.replace('boxScoreHustle_', '')

df.columns = [rename(c) for c in df.columns]

# ── Split game-level vs player cols ──────────────────────────────────────────
is_player = df.columns.str.match(r'^.+\.(homeTeam|awayTeam)\.\d+$')
game_cols   = df.columns[~is_player].tolist()
player_cols = df.columns[is_player].tolist()

games = df[game_cols].copy()
print(f"games shape:   {games.shape}")
print(f"games columns: {game_cols}\n")

# ── wide_to_long ──────────────────────────────────────────────────────────────
# stubs = unique stat names (everything before the first '.')
stubs = list({c.split('.')[0] for c in player_cols})

players = pd.wide_to_long(
    df[['gameId'] + player_cols],
    stubnames=stubs,
    i='gameId',
    j='side_slot',
    sep='.',
    suffix=r'(?:homeTeam|awayTeam)\.\d+',
).reset_index()

# Split side_slot → side + slot
players[['side', 'slot']] = players['side_slot'].str.rsplit('.', n=1, expand=True)
players['slot'] = players['slot'].astype(int)
players = players.drop(columns='side_slot')

# Use nullable Int64 so NaN rows don't crash
players['personId'] = players['personId'].astype('Int64')

# Reorder for readability
front = ['gameId', 'side', 'slot', 'personId', 'firstName', 'familyName',
         'nameI', 'playerSlug', 'position', 'comment', 'jerseyNum', 'minutes', 'points']
rest  = [c for c in players.columns if c not in front]
players = players[front + rest]

print(f"player_stats shape:   {players.shape}")
print(f"player_stats columns: {list(players.columns)}\n")

# ── Save ──────────────────────────────────────────────────────────────────────
# games.to_parquet(OUT_DIR + 'HustleGames.parquet', index=False)
# players.to_parquet(OUT_DIR + 'HustlePlayerStats.parquet', index=False)
print("Saved:")
print(f"  {OUT_DIR}HustleGames.parquet")
print(f"  {OUT_DIR}HustlePlayerStats.parquet")

games shape:   (18247, 47)
games columns: ['gameId', 'awayTeamId', 'homeTeamId', 'homeTeam_teamId', 'homeTeam_teamCity', 'homeTeam_teamName', 'homeTeam_teamTricode', 'homeTeam_teamSlug', 'homeTeam_statistics_minutes', 'homeTeam_statistics_points', 'homeTeam_statistics_contestedShots', 'homeTeam_statistics_contestedShots2pt', 'homeTeam_statistics_contestedShots3pt', 'homeTeam_statistics_deflections', 'homeTeam_statistics_chargesDrawn', 'homeTeam_statistics_screenAssists', 'homeTeam_statistics_screenAssistPoints', 'homeTeam_statistics_looseBallsRecoveredOffensive', 'homeTeam_statistics_looseBallsRecoveredDefensive', 'homeTeam_statistics_looseBallsRecoveredTotal', 'homeTeam_statistics_offensiveBoxOuts', 'homeTeam_statistics_defensiveBoxOuts', 'homeTeam_statistics_boxOutPlayerTeamRebounds', 'homeTeam_statistics_boxOutPlayerRebounds', 'homeTeam_statistics_boxOuts', 'awayTeam_teamId', 'awayTeam_teamCity', 'awayTeam_teamName', 'awayTeam_teamTricode', 'awayTeam_teamSlug', 'awayTeam_statistics_

In [10]:
import pandas as pd

SRC     = '/Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data/CRUDEHustle.parquet'
GAMES   = '/Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data/HustleGames.parquet'
PLAYERS = '/Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data/HustlePlayerStats.parquet'

src     = pd.read_parquet(SRC)
games   = pd.read_parquet(GAMES)
players = pd.read_parquet(PLAYERS)

# ── Column count check ────────────────────────────────────────────────────────
meta_dropped  = len([c for c in src.columns if c.startswith('meta_')])
src_remaining = len(src.columns) - meta_dropped

# games has game-level cols
# players has gameId (shared) + side + slot (new) + all player stat cols
# subtract 1 for gameId overlap, subtract 2 for side/slot which are new
unique_output_cols = len(games.columns) + len(players.columns) - 1 - 2

print("── Column audit ─────────────────────────────────────────────────────")
print(f"  Source columns:           {len(src.columns)}")
print(f"  meta_* dropped:           {meta_dropped}")
print(f"  Source remaining:         {src_remaining}")
print(f"  games cols:               {len(games.columns)}")
print(f"  player_stats cols:        {len(players.columns)}")
print(f"  gameId overlap:           -1")
print(f"  side/slot added (new):    -2")
print(f"  Unique output cols:       {unique_output_cols}")
print(f"  Match: {'✓' if unique_output_cols == src_remaining else '✗ MISMATCH'}")

# ── Row check ─────────────────────────────────────────────────────────────────
print("\n── Row audit ────────────────────────────────────────────────────────")
print(f"  Source rows:              {len(src)}")
print(f"  games rows:               {len(games)}  (should equal source rows)")
print(f"  player_stats rows:        {len(players)}")
avg_players = len(players) / len(src)
print(f"  Avg players per game:     {avg_players:.1f}  (expect ~20)")

# ── gameId coverage ───────────────────────────────────────────────────────────
print("\n── gameId coverage ──────────────────────────────────────────────────")
src_ids     = set(src['boxScoreHustle_gameId'].unique())
games_ids   = set(games['gameId'].unique())
players_ids = set(players['gameId'].unique())
print(f"  Unique gameIds in source:       {len(src_ids)}")
print(f"  Unique gameIds in games:        {len(games_ids)}")
print(f"  Unique gameIds in player_stats: {len(players_ids)}")
print(f"  games covers all source games:        {'✓' if src_ids == games_ids else '✗ MISMATCH'}")
print(f"  player_stats covers all source games: {'✓' if src_ids == players_ids else '✗ MISMATCH'}")

# ── Null check ────────────────────────────────────────────────────────────────
print("\n── Null check (player_stats key fields) ─────────────────────────────")
for col in ['gameId', 'personId', 'side', 'firstName']:
    n = players[col].isna().sum()
    print(f"  {col}: {n} nulls")

print("\n── Sample player_stats row ──────────────────────────────────────────")
print(players[['gameId', 'side', 'slot', 'personId', 'firstName', 'familyName', 'minutes', 'deflections']].head(5).to_string(index=False))

── Column audit ─────────────────────────────────────────────────────
  Source columns:           1025
  meta_* dropped:           3
  Source remaining:         1022
  games cols:               47
  player_stats cols:        28
  gameId overlap:           -1
  side/slot added (new):    -2
  Unique output cols:       72
  Match: ✗ MISMATCH

── Row audit ────────────────────────────────────────────────────────
  Source rows:              18247
  games rows:               18247  (should equal source rows)
  player_stats rows:        711633
  Avg players per game:     39.0  (expect ~20)

── gameId coverage ──────────────────────────────────────────────────
  Unique gameIds in source:       18247
  Unique gameIds in games:        18247
  Unique gameIds in player_stats: 18247
  games covers all source games:        ✓
  player_stats covers all source games: ✓

── Null check (player_stats key fields) ─────────────────────────────
  gameId: 0 nulls
  personId: 424631 nulls
  side: 0 nulls
  fir

In [8]:
import os
import glob
import pyarrow.parquet as pq

directory_path = "/Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data/"

src_set = set(pq.ParquetFile(directory_path + "CRUDESummary.parquet").schema_arrow.names)

search_pattern = os.path.join(directory_path, "CRUDESummary*.parquet")
parquet_files = glob.glob(search_pattern)

# Union of ALL columns across all output files
all_output_cols = set()
for f in parquet_files:
    if "CRUDESummary.parquet" not in f:
        all_output_cols |= set(pq.ParquetFile(f).schema_arrow.names)

# Src cols that appear in none of the output files
missing = src_set - all_output_cols

print(f"Source columns:      {len(src_set)}")
print(f"Output files found:  {len(parquet_files) - 1}")
print(f"Covered by outputs:  {len(src_set & all_output_cols)}")
print(f"Missing from all:    {len(missing)}")
print("\nMissing columns:")
for col in sorted(missing):
    print(f"  {col}")

Source columns:      230
Output files found:  4
Covered by outputs:  18
Missing from all:    212

Missing columns:
  arena.arenaCity
  arena.arenaCountry
  arena.arenaId
  arena.arenaName
  arena.arenaPostalCode
  arena.arenaState
  arena.arenaStreetAddress
  arena.arenaTimezone
  attendance
  awayTeam.inBonus
  awayTeam.period_1_score
  awayTeam.period_1_type
  awayTeam.period_2_score
  awayTeam.period_2_type
  awayTeam.period_3_score
  awayTeam.period_3_type
  awayTeam.period_4_score
  awayTeam.period_4_type
  awayTeam.period_5_score
  awayTeam.period_5_type
  awayTeam.period_6_score
  awayTeam.period_6_type
  awayTeam.period_7_score
  awayTeam.period_7_type
  awayTeam.seed
  awayTeam.teamLosses
  awayTeam.teamWins
  awayTeam.timeoutsRemaining
  awayTeamId
  duration
  gameCode
  gameLabel
  gameSubLabel
  gameSubtype
  historicalStatus
  homeTeam.inBonus
  homeTeam.period_1_score
  homeTeam.period_1_type
  homeTeam.period_2_score
  homeTeam.period_2_type
  homeTeam.period_3_score
  

In [4]:
import pandas as pd

df=pd.read_parquet('/Users/michaelharoon/Projects/prediction_markets/nba/data_curation/data/BoxScoresSummaryOtherStats.parquet')

OSError: Couldn't deserialize thrift: don't know what type: 
Deserializing page header failed.
